# Silver → Gold: ontology-bound projections

All domain mappings come from the expanded portable ontology supplied by deployment. Exact integer cents and source columns are retained; human-facing monetary aliases are decimals. Projections use the original Silver columns and ANSI casts. Required source values and entity keys are checked after casting; nullable fields remain nullable. Payment cents are reconciled by firm and currency before any Gold write. This projection reconciliation does not replace the generator and Silver source invariants.

Filtered relationship tables make nullable adjustment targets explicit without multiplying financial rows. MatterAccess expresses policy but does **not** install authorization: these provider Gold tables remain administrator-only until a supported customer data-plane security gate is implemented. No global partner aggregates are published. The seven measure SQL definitions stay in the portable contract; the current Fabric public ontology definition has no native measure DSL.

Validation status: offline notebook-format, Python AST, deployment-payload, projection-construction, and lint checks only. No Fabric execution has been performed; symbolic projection tests do not execute Spark SQL, validate cast results, install permissions, or establish Delta runtime success.


In [ ]:
# Deployment parameter cell; replaced in memory before upload.
CONFIG_JSON = '{}'


In [ ]:
import json
import re
from datetime import date
from time import perf_counter
from uuid import UUID

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.conf.set('spark.sql.ansi.enabled', 'true')
config = json.loads(CONFIG_JSON)
required = {'source_workspace_id', 'source_lakehouse_id', 'target_workspace_id',
            'target_lakehouse_id', 'firm_slug', 'tables', 'schemas', 'ontology', 'as_of'}
if not required.issubset(config):
    raise ValueError('Run through fabric/deploy.py with the expanded ontology contract')
for key in ('source_workspace_id', 'source_lakehouse_id', 'target_workspace_id', 'target_lakehouse_id'):
    UUID(config[key])
if config['source_workspace_id'] != config['target_workspace_id']:
    raise ValueError('Provider transformation must remain in its provider workspace')
firm = config['firm_slug']
if not re.fullmatch(r'[a-z]+', firm):
    raise ValueError('Invalid firm slug')
snapshot = date.fromisoformat(config['as_of'])
source = f"abfss://{config['source_workspace_id']}@onelake.dfs.fabric.microsoft.com/{config['source_lakehouse_id']}"
target = f"abfss://{config['target_workspace_id']}@onelake.dfs.fabric.microsoft.com/{config['target_lakehouse_id']}"
if source == target:
    raise ValueError('Source and target must differ')
contract = config['ontology']
if contract.get('version') != 1:
    raise ValueError('Unsupported portable ontology version')
entities = {e['table']: e for e in contract['entities']}
frames = {}
counts = {}
started = perf_counter()
for table in config['tables']:
    if not re.fullmatch(r'[a-z_]+', table):
        raise ValueError('Invalid table name')
    raw = spark.read.format('delta').load(f'{source}/Tables/{firm}_{table}')
    if raw.where(F.col('firm_id').isNull() | (F.col('firm_id') != F.lit(firm + '_f001'))).limit(1).count():
        raise ValueError(f'Cross-firm or missing firm row in {table}')
    spec = config['schemas'][table]
    source_columns = {field['name'] for field in spec['spark_schema']['fields']}
    if not source_columns.issubset(raw.columns):
        raise ValueError(f'Missing Silver source columns: {table}')
    entity = entities.get(table, {})
    properties = entity.get('properties', {})
    replaced = {prop['column'] for prop in properties.values()}
    selected = [F.col(column) for column in raw.columns if column not in replaced]
    selected.extend(F.expr(prop.get('expression', '`' + prop['column'] + '`')).cast(prop['type']).alias(prop['column'])
                    for prop in properties.values())
    frame = raw.select(*selected).cache()
    counts[table] = frame.count()
    if counts[table] != raw.count():
        raise ValueError(f'Projection multiplied rows: {table}')
    key = spec['primary_key']
    # Validate after casts; nullable dates and relationship targets must remain nullable.
    required_columns = {field['name'] for field in spec['spark_schema']['fields'] if not field['nullable']}
    required_columns.add(key)
    if entity:
        required_columns.add(properties[entity['key']]['column'])
    for column in sorted(required_columns):
        if frame.where(F.col(column).isNull()).limit(1).count():
            raise ValueError(f'Required Gold value missing after cast: {table}.{column}')
    if frame.groupBy(key).count().where('count > 1').limit(1).count():
        raise ValueError(f'Invalid Gold key: {table}')
    if table == 'payments':
        # Reconcile exact cash per firm/currency before any Gold table is written.
        cash = F.sum(F.col('amount_cents').cast('decimal(38,0)')).alias('cash_cents')
        source_cash = raw.groupBy('firm_id', 'currency').agg(cash)
        gold_cash = frame.groupBy('firm_id', 'currency').agg(cash)
        if source_cash.exceptAll(gold_cash).limit(1).count() or gold_cash.exceptAll(source_cash).limit(1).count():
            raise ValueError('Gold cash reconciliation failed')
    frames[table] = frame
for entity in contract['entities']:
    if entity['table'] not in frames:
        raise ValueError(f"Missing entity source {entity['table']}")
for relationship in contract['relationships']:
    frame = frames[relationship['table']]
    if relationship.get('filter'):
        columns = list(dict.fromkeys(['firm_id', relationship['source_column'], relationship['target_column']]))
        edge = frame.where(relationship['filter']).select(*columns).dropDuplicates().cache()
        frames[relationship['edge_table']] = edge
        counts[relationship['edge_table']] = edge.count()
for table, frame in frames.items():
    location = f'{target}/Tables/{table}'
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(location)
    if spark.read.format('delta').load(location).count() != counts[table]:
        raise ValueError(f'Persisted Gold row count mismatch: {table}')
    frame.unpersist()
print(json.dumps({'stage': 'silver_to_gold', 'firm': firm, 'as_of': str(snapshot),
                  'rows': counts, 'ontology_entities': len(entities),
                  'authorization_installed': False,
                  'elapsed_seconds': round(perf_counter() - started, 3)}))
